# 005 · break, continue and Loop else

> **Syntax › Control Flow** · Python 3.14

`break` and `continue` work exactly as in C#. Python adds something C# doesn't have — an `else` clause on loops that runs only if the loop **wasn't** broken out of — and lacks something C# does have: `goto` and any form of labelled break.

**In this notebook you will learn:**

- `break` and `continue` in `for` and `while` loops
- `for … else` / `while … else` and the "search" idiom it replaces
- why Python has no `goto` or labelled `break`
- how to break out of nested loops: flags, `return` from a function, exceptions, and flattening with `itertools.product`

## C# mental model

| C# | Python |
|---|---|
| `break;` | `break` |
| `continue;` | `continue` |
| `bool found = false; ... if (!found) { }` | `for ... else:` |
| `goto label;` | — (no `goto`) |
| `goto` out of nested loops | flag, `return`, exception, or flatten the loops |

## `break` — leave the loop

```csharp
// C#
foreach (var n in numbers)
{
    if (n < 0) break;
    Console.WriteLine(n);
}
```

In [ ]:
numbers = [4, 8, 15, -1, 16, 23]

for n in numbers:
    if n < 0:
        print("negative value found, stopping")
        break
    print(n)

## `continue` — skip to the next iteration

In [ ]:
orders = [{"id": 1, "status": "paid"}, {"id": 2, "status": "cancelled"}, {"id": 3, "status": "paid"}]

for order in orders:
    if order["status"] == "cancelled":
        continue                      # skip the rest of this iteration
    print("shipping order", order["id"])

`continue` is often used as a guard clause inside a loop body, keeping the main logic un-indented *(see Control Flow › 001 · if / elif / else)*. In `while` loops, make sure the loop variable is updated **before** `continue`, or you'll loop forever:

In [ ]:
i = 0
while i < 6:
    i += 1                            # update first — a `continue` above this line would loop forever
    if i % 2 == 0:
        continue
    print("odd:", i)

## Loop `else` — "no break happened"

A `for` or `while` loop can have an `else` block. It runs when the loop finishes **normally** (the iterable is exhausted or the `while` condition becomes false) and is **skipped** if the loop exits via `break`.

The classic use is a **search**: `break` when you find it; the `else` handles "not found". In C# you'd need a flag:

```csharp
// C#
bool found = false;
foreach (var user in users)
{
    if (user == target)
    {
        Console.WriteLine($"found {user}");
        found = true;
        break;
    }
}
if (!found) Console.WriteLine("not found");
```

In [ ]:
def find_user(users: list[str], target: str) -> None:
    for user in users:
        if user == target:
            print(f"found {user}")
            break
    else:                             # runs only if the loop did NOT break
        print(f"{target} not found")

users = ["alice", "bob", "carol"]
find_user(users, "bob")
find_user(users, "dave")

Another example — checking for primes. The `else` belongs to the `for`, not the `if` (look at the indentation):

In [ ]:
for n in range(2, 20):
    for divisor in range(2, int(n ** 0.5) + 1):
        if n % divisor == 0:
            break                     # n is composite
    else:
        print(n, end=" ")             # no divisor found -> prime
print()

`while` supports `else` too, with the same rule:

In [ ]:
attempts = 0
while attempts < 3:
    attempts += 1
    succeeded = attempts == 5         # pretend the operation never succeeds
    if succeeded:
        print("connected")
        break
else:
    print(f"gave up after {attempts} attempts")

⚠️ **Gotcha:** the name `else` is misleading — it doesn't mean "if the loop didn't run". It also runs when the iterable is **empty**, because no `break` happened. A mental rename to `nobreak:` helps.

In [ ]:
for item in []:
    print("never printed")
else:
    print("else ran — the loop was empty, but there was no break")

**Tip:** for simple searches, a built-in is often clearer than `for … else`: `x in items`, `any(...)`, or `next(..., default)` *(see Iterators & Generators › 003 · Generator Expressions and Lazy Evaluation)*.

In [ ]:
users = ["alice", "bob", "carol"]

print("bob" in users)
print(any(u.startswith("c") for u in users))
print(next((u for u in users if len(u) > 4), "nobody"))   # first match or a default

## No labelled `break` and no `goto`

C# can exit nested loops with `goto`. Java and JavaScript have labelled `break`. Python has **neither** — a `break` only ever exits the **innermost** loop. (A proposal for labelled breaks, PEP 3136, was rejected.)

```csharp
// C#
for (int r = 0; r < grid.Length; r++)
    for (int c = 0; c < grid[r].Length; c++)
        if (grid[r][c] == target) goto Found;
Console.WriteLine("not found");
Found:
    ...
```

In [ ]:
grid = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

for row in grid:
    for value in row:
        if value == 5:
            break                     # only exits the inner loop
    print("finished row", row)        # the outer loop keeps going

Here are the idiomatic alternatives, from most to least preferred.

### 1. Put the loops in a function and `return`

Usually the cleanest: the search becomes a named, testable function.

In [ ]:
def find_in_grid(grid: list[list[int]], target: int) -> tuple[int, int] | None:
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            if value == target:
                return r, c           # exits both loops at once
    return None

print(find_in_grid(grid, 5))
print(find_in_grid(grid, 42))

### 2. Flatten the loops with `itertools.product`

If you're iterating over the Cartesian product of ranges, a single loop means a single `break`:

In [ ]:
from itertools import product

for r, c in product(range(len(grid)), range(len(grid[0]))):
    if grid[r][c] == 5:
        print(f"found at ({r}, {c})")
        break
else:
    print("not found")

### 3. A flag variable

The C#-style approach. It works but is noisy:

In [ ]:
found = None
for r, row in enumerate(grid):
    for c, value in enumerate(row):
        if value == 8:
            found = (r, c)
            break
    if found is not None:             # second break for the outer loop
        break
print("found:", found)

### 4. `for … else` + `continue` trick

The inner loop's `else` runs `continue` (skipping the outer `break`) unless the inner loop broke. Clever, but many readers find it confusing — prefer a function.

In [ ]:
for r, row in enumerate(grid):
    for c, value in enumerate(row):
        if value == 6:
            break
    else:
        continue                      # inner loop didn't break -> next row
    break                             # inner loop broke -> break the outer loop too
print(f"found at ({r}, {c})")

### 5. Raise and catch an exception

Possible, but exceptions for normal control flow are best reserved for deep call stacks *(see Error Handling › 002 · Raising and Custom Exceptions)*:

In [ ]:
class Found(Exception):
    pass

try:
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            if value == 9:
                raise Found
except Found:
    print(f"found at ({r}, {c})")

## `break` and `continue` in `try`/`finally`

As in C#, a `finally` block still runs when `break` or `continue` leaves a `try` block *(see Error Handling › 001 · try / except / finally)*:

In [ ]:
for i in range(3):
    try:
        if i == 1:
            break
        print("working on", i)
    finally:
        print("cleanup for", i)

## Summary

| Task | C# | Python |
|---|---|---|
| Exit loop | `break;` | `break` |
| Next iteration | `continue;` | `continue` |
| "Not found" after search | flag variable | `for ... else:` |
| Exit nested loops | `goto` / flag | `return` from a function (best), flag, `product`, exception |
| Find first match | `FirstOrDefault(pred)` | `next((x for x in xs if pred(x)), default)` |
| Any match | `Any(pred)` | `any(pred(x) for x in xs)` |

## Exercises

1. Loop over `words = ["alpha", "beta", "", "gamma"]` and print each word, stopping at the first empty string. Print `"all words processed"` only if no empty string was found (use loop `else`).
2. Print the numbers 1–20 that are divisible by neither 3 nor 5, using `continue`.
3. Write `first_duplicate(items)` that returns the first value that appears twice, or `None`. Use nested loops and `return` to exit both.
4. What does this print, and why?
   ```python
   for x in range(3):
       pass
   else:
       print("done", x)
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
words = ["alpha", "beta", "", "gamma"]
for word in words:
    if not word:
        break
    print(word)
else:
    print("all words processed")

# 2
for n in range(1, 21):
    if n % 3 == 0 or n % 5 == 0:
        continue
    print(n)

# 3 — O(n²); a set-based version would be O(n)
def first_duplicate(items):
    for i, a in enumerate(items):
        for b in items[i + 1:]:
            if a == b:
                return a
    return None

# 4 — prints "done 2": there was no break, so else runs,
#     and the loop variable x still exists after the loop.
```

</details>